In [ ]:
import torch
import math

In [820]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [ ]:

class Head(torch.nn.Module):

    def __init__(self, block_size, d_model, d_v, d_k):
        super().__init__()
        self.d_k = d_k
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_v)
        self.register_buffer("causal_mask", self.ins_causal_mask(block_size))


    def forward(self, X_e):
        B, T, d_model = X_e.shape
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_v)
        QK = Q @ K.transpose(1,2)
        QK_scale = QK/math.sqrt(self.d_k) #(B,T,T)
        QK_scale += self.causal_mask[:T, :T]
        QV_softmax = torch.nn.functional.softmax(QK_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V #(B, T, d_v)
        return added_weights


    @staticmethod
    def ins_causal_mask(block_size):
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


class MLP(torch.nn.Module):

    def __init__(self, d_model):
            
        super().__init__()
        self.W_ff1 = torch.nn.Linear(d_model, 4*d_model)
        self.W_ff2 = torch.nn.Linear(4*d_model, d_model)

    def forward(self, X):

        linear_1 = self.W_ff1(X)
        non_linearity = torch.nn.functional.gelu(linear_1)
        linear_2 = self.W_ff2(non_linearity)
        return linear_2

        

class MultiHead(torch.nn.Module):
    
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        assert d_model % n_heads == 0
        d_v = d_model // n_heads
        d_k = d_k if d_k is not None else d_v
        self.heads = torch.nn.ModuleList([Head(block_size, d_model, d_v, d_k) for _ in range(n_heads)])
        self.W_o = torch.nn.Linear(d_model, d_model)

    def forward(self, X_e):

        added_weights = [head(X_e) for head in self.heads]
        concat = torch.cat(added_weights, dim = -1)
        return self.W_o(concat)

class Block(torch.nn.Module):
     
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        self.att = MultiHead(block_size, d_model, n_heads, d_k)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.mlp = MLP(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)

    def forward(self, X):

        X = X + self.att(self.ln_1(X))
        X = X + self.mlp(self.ln_2(X))
        return X


class Decoder_Only_Transformer(torch.nn.Module):

    def __init__(self, vocab_size, block_size, d_model, n_heads = 1, d_k = None, n_layers = 1):

        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model) 
        self.blocks = torch.nn.ModuleList([Block(block_size, d_model, n_heads, d_k) for _ in range(n_layers)])
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)
        self.register_buffer("pos_encoding", self.sinusoidal_pe(block_size, d_model))
        self.loss = torch.nn.CrossEntropyLoss()

    
    @staticmethod
    def sinusoidal_pe(block_size, d_model):
        pe = torch.zeros(block_size, d_model)
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denom)
                pe[pos, i + 1] = math.cos(pos / denom)
        return pe

    
    
    def forward(self, X, Y = None):

        B, T = X.shape
        X_e = self.embedding(X) + self.pos_encoding[:T] #(B,T,d_model)
        for block in self.blocks:
            X_e = block(X_e)
        X_e = self.ln_f(X_e) #(B,T,d_model)
        logits = self.linear(X_e) #(B,T,vocab_size)
        loss = None
        if Y is not None:
            loss = self.loss(logits.reshape(B*T,-1), Y.reshape(B*T,))
        
        return logits, loss




In [ ]:
def get_batch(data, batch_size, block_size):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch(train_data, batch_size, block_size)

In [818]:
permited = set("abcdefghijklmnñopqrstuvwxyzáéíóúü"
                 "ABCDEFGHIJKLMNÑOPQRSTUVWXYZÁÉÍÓÚÜ"
                 " \n.,;:¡!¿?-'\"()") #clean the noisy chars

In [ ]:
# 0. Variables and config

train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 128
batch_size = 32
d_model = 128
n_heads	= 4
n_layers = 4
torch.manual_seed(0)

# 1. Cleansed text
text = ''.join(c for c in text if c in permited)

# 2. Vocabulary out of the cleansed text
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

# 3. Encode
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, test_data = data[:n], data[n:]

# 4. Model
model = decoder_only_transformer(vocab_size, block_size, d_model, n_heads = n_heads, n_layers = n_layers)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 5. Health checks
assert data.max().item() < vocab_size, (data.max().item(), vocab_size)
print("vocab:", vocab_size, "| tokens:", len(data))

vocab: 80 | tokens: 2066505


In [868]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    model.eval()
    out = {}
    for split, data in [("train", train_data), ("val", test_data)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(data, batch_size, block_size)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [875]:
x = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(x)

813904


In [869]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
max_steps = 10000
eval_interval = 1000
for step in range(max_steps):
    if step % eval_interval == 0:
        losses = estimate_loss(model)
        print(f"step {step}: train {losses['train']:.3f}, val {losses['val']:.3f}")

    x, y = get_batch(train_data, batch_size, block_size)
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train 4.591, val 4.610
step 1000: train 1.750, val 1.733
step 2000: train 1.530, val 1.561
step 3000: train 1.400, val 1.457
step 4000: train 1.331, val 1.405
step 5000: train 1.266, val 1.359
step 6000: train 1.232, val 1.333
step 7000: train 1.213, val 1.305
step 8000: train 1.185, val 1.287
step 9000: train 1.174, val 1.271


In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()
    block_size = model.pos_encoding.shape[0]

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = torch.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)  # (B, 1)
        idx = torch.cat([idx, next_tok], dim=1)
    return idx

In [ ]:
decode = lambda l: ''.join([itos[i] for i in l])

idx = torch.zeros((1, 1), dtype=torch.long)
out = generate(model, idx, max_new_tokens=500, temperature=0.8, top_k=20)
print(decode(out[0].tolist()))

prompt = torch.tensor([encode("ROMEO:\n")], dtype=torch.long)
out = generate(model, prompt, max_new_tokens=300, temperature=0.8)
print(decode(out[0].tolist()))

In [872]:
torch.save({
    "model": model.state_dict(),
    "config": dict(vocab_size=vocab_size, block_size=block_size, d_model=d_model,
                   n_heads=n_heads, n_layers=n_layers),
    "chars": chars,
}, "lorca.pt")


In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)